<a href="https://colab.research.google.com/github/jacobgreen4477/The-4th-ETRI-AI-Human-Understanding-Competition/blob/main/etri_baseline_v5_0_4(%EC%A6%9D%EA%B0%95).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> title : 122_etri_lifelog_dm_llm-impute_vF <br>
 -  코드 실행 전 PATH 변경하세요.
 - 코드 실행 전 vllm 패키지를 설치하세요.
 - 코드 실행 전 Qwen/Qwen3-8B 모델을 다운로드 하세요.

### 🔨 PATH 설정

In [1]:
from google.colab import drive, files
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
PATH  =  '/content/drive/MyDrive/data/ch2025_data_items/share/submissions/input'

### 📦 vllm 설치 코드

In [1]:
%%time

"""
vllm 설치 코드
"""

!pip install -U langchain-community  >/dev/null
!pip install bitsandbytes >/dev/null
!pip install -U transformers accelerate >/dev/null
!pip install faiss-gpu-cu12 --no-deps >/dev/null
!pip install datasets >/dev/null
!pip install vllm >/dev/null
!pip install --upgrade transformers >/dev/null

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
CPU times: user 11.3 s, sys: 1.98 s, total: 13.3 s
Wall time: 3min 24s


### 📦 LLM 다운로드 코드

In [ ]:
%%time

"""
LLM 다운로드 코드
"""
# from transformers import AutoModelForCausalLM, AutoTokenizer
# import os

# # 구글 드라이브 경로
# drive_path = "/content/drive/MyDrive/models2"

# # 모델명
# model_id = "Qwen/Qwen3-8B"

# # 저장 경로
# save_path = os.path.join(drive_path, model_id)

# # 토크나이저와 모델 다운로드 (Drive에 자동 저장됨)
# tokenizer = AutoTokenizer.from_pretrained(model_id, cache_dir=save_path)
# model = AutoModelForCausalLM.from_pretrained(model_id, cache_dir=save_path)

# print(f"✅ 모델과 토크나이저가 저장되었습니다: {save_path}")

### 📦 다운로드 된 LLM 로드

In [3]:
%%time

"""
다운로드 된 LLM 로드
"""

import os
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

from vllm import LLM, SamplingParams

# 모델 저장 경로
drive_path = "/content/drive/MyDrive/models2/"

# 모델명
model_id   = 'Qwen/Qwen3-8B'

# vllm
llm = LLM(
    model=f"{drive_path}{model_id}",
    tokenizer=f"{drive_path}{model_id}",
    tensor_parallel_size=1,
    dtype="bfloat16",
    load_format="auto",
    gpu_memory_utilization=0.8,
    max_model_len=38960,
    enforce_eager=True,
)

from transformers import AutoTokenizer, AutoModelForCausalLM

# Tokenizer 로드
tokenizer = AutoTokenizer.from_pretrained("/content/drive/MyDrive/models2/Qwen/Qwen3-8B", trust_remote_code=True)

INFO 09-10 07:11:05 [__init__.py:241] Automatically detected platform cuda.
INFO 09-10 07:11:06 [utils.py:326] non-default args: {'model': '/content/drive/MyDrive/models2/Qwen/Qwen3-8B', 'tokenizer': '/content/drive/MyDrive/models2/Qwen/Qwen3-8B', 'dtype': 'bfloat16', 'max_model_len': 38960, 'gpu_memory_utilization': 0.8, 'disable_log_stats': True, 'enforce_eager': True}
INFO 09-10 07:11:26 [__init__.py:711] Resolved architecture: Qwen3ForCausalLM


`torch_dtype` is deprecated! Use `dtype` instead!


INFO 09-10 07:11:26 [__init__.py:1750] Using max model len 38960
INFO 09-10 07:11:28 [scheduler.py:222] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 09-10 07:11:28 [__init__.py:3565] Cudagraph is disabled under eager mode
INFO 09-10 07:17:24 [llm.py:298] Supported_tasks: ['generate']
CPU times: user 14.1 s, sys: 1.66 s, total: 15.7 s
Wall time: 6min 29s


### 📦 라이브러리

In [4]:
# Core Libraries
import os
import sys
import re
import ast
import glob
import random
from functools import reduce
from io import StringIO
from collections import Counter

# Numerical Operations
import numpy as np
import pandas as pd

# Progress Tracking
from tqdm import tqdm

# Warnings
import warnings
warnings.filterwarnings('ignore')

# seed 고정
SD = 42
random.seed(SD)
np.random.seed(SD)
os.environ['PYTHONHASHSEED'] = str(SD)

# pandas 옵션
pd.set_option('display.max_columns', 999)
pd.set_option('display.max_rows', 999)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.float_format', lambda x: '%0.4f' % x)

In [5]:
def preprocess_mScreenStatus(df):
    from datetime import datetime, timedelta

    df = df.copy()
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df['lifelog_date'] = pd.to_datetime(df['lifelog_date'])

    base_keys = df[['subject_id', 'lifelog_date']].drop_duplicates()
    base_keys['lifelog_date'] = base_keys['lifelog_date'].dt.date

    # 밤 9시 ~ 다음날 오전 11시 필터링
    df['hour'] = df['timestamp'].dt.hour
    df = df[(df['hour'] >= 21) | (df['hour'] < 11)].copy()
    df.loc[df['hour'] < 11, 'lifelog_date'] -= pd.Timedelta(days=1)
    df.sort_values(['subject_id', 'timestamp'], inplace=True)

    results = []

    for (subject_id, lifelog_date), group in df.groupby(['subject_id', 'lifelog_date']):
        group = group.sort_values('timestamp').reset_index(drop=True)

        # 중간 각성 제거
        prev = group['m_screen_use'].shift(1)
        next_ = group['m_screen_use'].shift(-1)
        mask = (group['m_screen_use'] == 1) & (prev == 0) & (next_ == 0)
        group.loc[mask, 'm_screen_use'] = 0

        # 짧은 각성 블록 제거
        group['is_sleep'] = group['m_screen_use'] == 0
        group['block'] = (group['is_sleep'] != group['is_sleep'].shift()).cumsum()
        block_info = group.groupby('block').agg(
            is_sleep=('is_sleep', 'first'),
            size=('is_sleep', 'size')
        )

        for i in range(1, len(block_info) - 1):
            if (
                block_info.iloc[i]['is_sleep'] == False and
                block_info.iloc[i]['size'] <= 2 and
                block_info.iloc[i - 1]['is_sleep'] and
                block_info.iloc[i + 1]['is_sleep']
            ):
                group.loc[group['block'] == block_info.index[i], 'm_screen_use'] = 0

        # 블록 재계산
        group['is_sleep'] = group['m_screen_use'] == 0
        group['block'] = (group['is_sleep'] != group['is_sleep'].shift()).cumsum()

        sleep_blocks = group[group['is_sleep']].groupby('block').agg(
            sleep_start=('timestamp', 'first'),
            sleep_end=('timestamp', 'last'),
            duration_min=('timestamp', lambda x: (x.max() - x.min()).total_seconds() / 60)
        )

        sleep_time = wake_time = duration_min = None
        if not sleep_blocks.empty:
            longest_sleep = sleep_blocks.loc[sleep_blocks['duration_min'].idxmax()]
            sleep_time = longest_sleep['sleep_start'].time()
            wake_time = longest_sleep['sleep_end'].time()
            duration_min = longest_sleep['duration_min']  # ✅ 정확하게 자정 넘는 경우도 반영됨

            # 유효 시간 범위 조건
            if not (4 <= wake_time.hour < 11):
                wake_time = None
            if not (sleep_time.hour >= 21 or sleep_time.hour < 3):
                sleep_time = None
            if duration_min < 100:
                sleep_time = None
                wake_time = None
                duration_min = None

        results.append({
            'subject_id': subject_id,
            'lifelog_date': lifelog_date.date(),
            'sleep_time': sleep_time,
            'wake_time': wake_time,
            'sleep_duration_min': round(duration_min, 1) if duration_min is not None else None
        })

    sleep_df = pd.DataFrame(results)
    result_df = base_keys.merge(sleep_df, on=['subject_id', 'lifelog_date'], how='left')

    # 시간 → 실수형 숫자 변환
    def time_to_float(t):
        if pd.isna(t):
            return None
        return round(t.hour + t.minute / 60 + t.second / 3600, 4)

    result_df['sleep_time'] = result_df['sleep_time'].apply(time_to_float)
    result_df['wake_time'] = result_df['wake_time'].apply(time_to_float)

    return result_df

In [6]:
def fill_missing_dates_by_subject(df, date_col='lifelog_date'):

    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])
    result = []

    for sid, group in df.groupby('subject_id'):
        group = group.sort_values(date_col)

        # 연속 날짜 생성
        full_dates = pd.date_range(start=group[date_col].min(), end=group[date_col].max())
        full_df = pd.DataFrame({date_col: full_dates})
        full_df['subject_id'] = sid

        # 병합
        merged = pd.merge(full_df, group, on=['subject_id', date_col], how='left')

        result.append(merged)

    # 병합 및 정렬
    final_df = pd.concat(result, ignore_index=True).sort_values(['subject_id', date_col])

    return final_df

In [7]:
def calculate_circular_mean_sleep_time(sleep_times):
    sleep_times = pd.Series(sleep_times).dropna()
    if len(sleep_times) == 0:
        return np.nan  # 혹은 return 0.0 등 기본값 설정 가능

    def hour_to_radian(hour):
        return (hour % 24) / 24 * 2 * np.pi

    radians = np.array([hour_to_radian(t) for t in sleep_times])
    mean_radian = np.arctan2(np.mean(np.sin(radians)), np.mean(np.cos(radians)))
    mean_hour = (mean_radian / (2 * np.pi)) * 24 % 24

    return mean_hour

### 📌 데이터 읽기

In [8]:
# 1
# mACStatus = pd.read_parquet(f'{PATH}/ETRI_lifelog_dataset/ch2025_data_items/ch2025_mACStatus.parquet')
# mActivity = pd.read_parquet(f'{PATH}/ETRI_lifelog_dataset/ch2025_data_items/ch2025_mActivity.parquet')
# mAmbience = pd.read_parquet(f'{PATH}/ETRI_lifelog_dataset/ch2025_data_items/ch2025_mAmbience.parquet')
# mBle = pd.read_parquet(f'{PATH}/ETRI_lifelog_dataset/ch2025_data_items/ch2025_mBle.parquet')
# mGps = pd.read_parquet(f'{PATH}/ETRI_lifelog_dataset/ch2025_data_items/ch2025_mGps.parquet')
# mLight = pd.read_parquet(f'{PATH}/ETRI_lifelog_dataset/ch2025_data_items/ch2025_mLight.parquet')
mScreenStatus = pd.read_parquet(f'{PATH}/ETRI_lifelog_dataset/ch2025_data_items/ch2025_mScreenStatus.parquet')
# mUsageStats = pd.read_parquet(f'{PATH}/ETRI_lifelog_dataset/ch2025_data_items/ch2025_mUsageStats.parquet')
# mWifi = pd.read_parquet(f'{PATH}/ETRI_lifelog_dataset/ch2025_data_items/ch2025_mWifi.parquet')
# wHr = pd.read_parquet(f'{PATH}/ETRI_lifelog_dataset/ch2025_data_items/ch2025_wHr.parquet')
# wLight = pd.read_parquet(f'{PATH}/ETRI_lifelog_dataset/ch2025_data_items/ch2025_wLight.parquet')
# wPedo = pd.read_parquet(f'{PATH}/ETRI_lifelog_dataset/ch2025_data_items/ch2025_wPedo.parquet')

# 2
train = pd.read_csv(f'{PATH}/ETRI_lifelog_dataset/ch2025_metrics_train.csv')
test = pd.read_csv(f'{PATH}/ETRI_lifelog_dataset/ch2025_submission_sample.csv')

### 📌 데이터 전처리

In [9]:
# lifelog_date
mScreenStatus['lifelog_date'] = mScreenStatus['timestamp'].astype(str).str[:10]

# 전처리코드 적용
mScreenStatus2 = preprocess_mScreenStatus(mScreenStatus)

# 결측일 생성
mScreenStatus2 = fill_missing_dates_by_subject(mScreenStatus2)

# weekday & month 컬럼 생성
weekday_map = {
    0: '월요일', 1: '화요일', 2: '수요일', 3: '목요일',
    4: '금요일', 5: '토요일', 6: '일요일'
}
mScreenStatus2['weekday'] = mScreenStatus2['lifelog_date'].dt.dayofweek.map(weekday_map)
mScreenStatus2['month'] = mScreenStatus2['lifelog_date'].dt.month

# TARGET 추가
mScreenStatus2['lifelog_date'] = mScreenStatus2['lifelog_date'].astype(str)
train['lifelog_date'] = train['lifelog_date'].astype(str)
mScreenStatus2 = mScreenStatus2.merge(train[['subject_id','lifelog_date','Q1','Q2','Q3','S1','S2','S3']],on=['subject_id','lifelog_date'],how='left')

# TARGET 컬럼명 추가설명
a1_map = {
    'sleep_duration_min':'수면시간(분)',
    'sleep_time':'취침시간',
    'wake_time':'기상시간',
    'Q1':'Q1(기상직후 수면의질)',
    'Q2':'Q2(취침직전 신체적피로)',
    'Q3':'Q3(취침직전 스트레스)',
    'S1':'S1(기상직후 수면시간)',
    'S2':'S2(기상직후 수면효율)',
    'S3':'S3(기상직후 수면지연시간)',
}
mScreenStatus2 = mScreenStatus2.rename(columns=a1_map)

# check
mScreenStatus2.head()

,lifelog_date,subject_id,취침시간,기상시간,수면시간(분),weekday,month,Q1(기상직후 수면의질),Q2(취침직전 신체적피로),Q3(취침직전 스트레스),S1(기상직후 수면시간),S2(기상직후 수면효율),S3(기상직후 수면지연시간)
0,2024-06-26,id01,23.4500,5.2500,348.0000,수요일,6,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000
1,2024-06-27,id01,23.1333,5.3000,370.0000,목요일,6,0.0000,0.0000,0.0000,0.0000,1.0000,1.0000
2,2024-06-28,id01,23.0000,5.9167,415.0000,금요일,6,1.0000,0.0000,0.0000,1.0000,1.0000,1.0000
3,2024-06-29,id01,21.8000,5.9167,487.0000,토요일,6,1.0000,0.0000,1.0000,2.0000,0.0000,0.0000
4,2024-06-30,id01,22.7000,5.1833,389.0000,일요일,6,0.0000,1.0000,1.0000,1.0000,1.0000,1.0000


### 📌 LLM 결측값 추정

In [ ]:
# 설정값
save_path = f'{PATH}/fillna'
os.makedirs(save_path, exist_ok=True)
dataname  = 'mScreenStatus'
version   = '20250607_vF'
data      = mScreenStatus2.copy()

In [ ]:
%%time

# CPU times: user 40.4 s, sys: 7.17 s, total: 47.6 s
# Wall time: 59min 11s

"""
- mACStatus: Indicates whether the smartphone is currently being charged.
- mActivity: Value calculated by the Google Activity Recognition API.
- mAmbience: Ambient sound identification labels and their respective probabilities.
- mBle: Bluetooth devices around individual subject.
- mGps: Multiple GPS coordinates measured within a single minute using the smartphone.
- mLight: Ambient light measured by the smartphone.
- (✔️) mScreenStatus: Indicates whether the smartphone screen is in use.
- mUsageStats: Indicates which apps were used on the smartphone and for how long.
- mWifi: Wifi devices around individual subject.
- wHr: Heart rate readings recorded by the smartwatch.
- wLight: Ambient light measured by the smartwatch.
- wPedo: Step data recorded by the smartwatch.
"""

system_message = f"""
# 🔈지침: 당신은 데이터 분석 전문가입니다.
- 모델 설명 : For the purpose of training a learning model to predict sleep health, fatigue, and stress, the following six metrics were derived from sleep sensor data and self-reported survey records.
- [데이터]에는 [취침시간], [기상시간] 결측치가 존재합니다. 이를 채워야 합니다.
- [취침시간]과 [기상시간]은 **24시간제를 기준으로 한 '소수 시간(decimal hour)' 형식**입니다.
    예) 23.50 → 23시 30분, 0.75 → 0시 45분, 1.0 → 1시 0분

# 🔍 평균값 설명
- 평균 취침시간/기상시간은 `21.0~2.0` 또는 `3.0~11.0` 범위에서 나타나며, 이는 **다음날로 넘어가는 원형 시간 범위입니다**.
    예) 1.4776은 1시 28분을 의미하며, 이는 24시를 넘어선 것이 아니라 **자정 이후 정상적인 시간대**입니다.

# 🔈결측치 보완 규칙
- 평균 취침시간/기상시간은 이미 올바른 포맷이며, 21.0~2.0(취침), 3.0~11.0(기상) 범위 내의 값입니다. **충분히 허용 가능한 값입니다. 혼란을 느끼지 말고 그대로 활용하세요.**
- 평균값이 0보다 작은 경우는 없으며, 1.4와 같은 값은 1시 24분을 의미합니다.
- 규칙 우선순위: (1) 범위 조건 > (2) 도메인 지식 > (3) 통계데이터 평균값
- 충돌 시, 무조건 통계데이터 값을 선택하라.
- 단, 범위를 벗어나면 가장 가까운 경계값으로 클리핑한다.
- 결론을 내리지 못하고 3회 이상 조건 검증을 반복하면, 즉시 가장 단순한 규칙(통계데이터 평균)으로 확정하라.

# 🔈금지 사항
- 데이터 형식이나 범위를 의심하거나, 평균값이 허용 범위를 벗어난다고 가정하지 마세요.
- 반복적으로 시간 포맷을 재해석하거나 '이 값이 잘못된 것 같다'는 내적 판단을 하지 마세요.

# 🔈주의사항 (중요!!)
- [취침시간] 결측값 추정 시, 21.0 ~ 2.0 범위 내의 값을 사용해야 합니다.
- [기상시간] 결측값 추정 시, 3.0 ~ 11.0 범위 내의 값을 사용해야 합니다.
- [취침시간], [기상시간] 결측값을 추정할 때 다양한 정보를 종합적으로 고려해야 합니다.
  → 다양한 정보: 전일 학습데이터, 통계데이터, Q1~S1 정보, 주말 유무(금요일 포함), 7~8월 유무 등
- [취침시간], [기상시간]의 **기존 값이 존재하는 경우 절대로 수정하지 마세요.**
- 기존값이 있는 경우에는 **해당 셀을 그대로 유지**하며, **빈칸(null)**인 경우에만 보완합니다.
- 입력으로 주어진 테이블의 다른 값들도 절대 수정하지 마세요. **오직 결측치만 채워야 합니다.**
- 누락된 셀은 반드시 채워야 하며, **빈 셀 없이 모든 셀을 채워진 상태로 출력**해야 합니다.
- 출력은 항상 **모든 셀에 값이 채워진 상태**여야 합니다. **빈칸을 남기지 마세요.**

### 🔈답변 작성 양식
- 답변에 지침내용을 포함하지 않습니다.
- 결측치만 채우고 기존값이 존재하는을 그대로 사용합니다.
"""

# run
parsed_results = []
for subject_id in tqdm(data['subject_id'].unique(), desc="Processing each subject"):
# for subject_id in tqdm(['id06'], desc="Processing each subject"):

    print(f'# subject_id:{subject_id}')

    sub1 = data[data['subject_id'] == subject_id]
    sub1 = sub1.drop(columns=['수면시간(분)'])
    학습데이터 = sub1.to_csv(index=False, sep="\t")

    # ----------------------------------------------------------------------------------------------
    # 통계데이터
    a1 = data[data['subject_id'] == subject_id].groupby(['weekday']).apply(lambda x:pd.Series({
    '평균취침시간': calculate_circular_mean_sleep_time(x['취침시간'])
    ,'평균기상시간': calculate_circular_mean_sleep_time(x['기상시간'])
    })).reset_index()

    a2 = data[data['subject_id'] == subject_id].groupby(['month','weekday']).apply(lambda x:pd.Series({
    '평균취침시간': calculate_circular_mean_sleep_time(x['취침시간'])
    ,'평균기상시간': calculate_circular_mean_sleep_time(x['기상시간'])
    })).reset_index()

    # a1을 weekday 기준으로 merge
    a2_filled = a2.merge(
        a1,
        on='weekday',
        suffixes=('', '_a1'),
        how='left'
    )

    # 결측값이 있는 경우 a1 값으로 대체
    a2_filled['평균취침시간'] = a2_filled['평균취침시간'].fillna(a2_filled['평균취침시간_a1'])
    a2_filled['평균기상시간'] = a2_filled['평균기상시간'].fillna(a2_filled['평균기상시간_a1'])

    # 보조 컬럼 제거
    sub2 = a2_filled.drop(columns=['평균취침시간_a1', '평균기상시간_a1'])
    통계데이터 = sub2.to_csv(index=False, sep="\t")
    # ----------------------------------------------------------------------------------------------

    user_message = f"""
    # 🔈작업 순서
    1. 결측치 [취침시간]을 추정하시오.
    2. 결측치 [기상시간]을 추정하시오 (추가지침: 전 단계에서 생성 된 [취침시간]을 참고해서 [기상시간] 결측값을 추정하시오.)

    # 🔈 xdata 설명
    - mScreenStatus(Indicates whether the smartphone screen is in use)기반으로 파생된 추정 [취침시간], [기상시간]
    - [취침시간]과 [기상시간]은 소수점으로 표현된 24시간제 시간입니다. (예: 22.6500 → 22시 39분)

    # 🔈 ydata 설명 (예측타겟)
    - Q1: 기상 직후 수면의 질 (0: 평균 이하, 1: 평균 이상)
    - Q2: 취침 직전 신체 피로 수준 (0: 높은 피로, 1: 낮은 피로)
    - Q3: 취침 직전 스트레스 수준 (0: 높은 스트레스, 1: 낮은 스트레스)
    - S1: 수면시간 가이드라인 준수여부 (0: 미준수, 1: 부분 준수, 2: 완전 준수)
    - S2: 수면 효율 가이드라인 준수여부 (0: 미준수, 1: 준수)
    - S3: 수면 잠들기 지연시간 가이드라인 준수여부 (0: 미준수, 1: 준수)

    # 🔈도메인 지식
    - S1(수면시간)이 2(완전준수)인 경우 -> 기상기상이 평소보다 늦어서 수면시간이 충분한 경우
    - 금요일, 토요일에는 다음날이 휴일이기 때문에 기상시간을 평소보다 늦게 설정
    - 7월, 8월에는 무더위(여름철 고온 시기)에는 기상시간이 일반적으로 더 빨라지는 경향 존재
    - 전일 상태가 오늘 상태에 영향을 줄 수 있음 (전일 몸이 좋지 않으면 다음날도 몸이 좋지 않는 것과 동일한 원리)

    # 🔈통계데이터
    {통계데이터}

    # 🔈데이터
    {학습데이터}

    # 🔈답변 출력 형식
    lifelog_date\tsubject_id\t취침시간\t기상시간\n
    2024-06-26\tid01\t23.4500\t5.2500\n
    2024-06-27\tid01\t23.1333\t5.3000\n
    2024-06-28\tid01\t23.0000\t5.9167\n

    # 답변:
    """

    # 최대 3회 시도
    for attempt in range(3):
        try:
            # 질의
            messages = [
                {"role": "system", "content": system_message},
                {"role": "user", "content": user_message}
            ]

            sampling_params = SamplingParams(max_tokens=37000, temperature=0, seed=42)
            outputs = llm.chat(messages, sampling_params=sampling_params)

            # 텍스트 추출 및 저장
            result_text = outputs[0].outputs[0].text
            with open(f"{save_path}/{dataname}_{subject_id}_result_try{attempt+1}.txt", "w", encoding="utf-8") as f:
                f.write(result_text)

            # <think> 제거
            cleaned_text = re.sub(r"<think>.*?</think>", "", result_text, flags=re.DOTALL).strip()

            # 파싱
            df_parsed = pd.read_csv(StringIO(cleaned_text), sep="\t")

            # 결측이 많을 경우 재시도
            # 1. 결측값이 10개 넘거나
            # 2. 생성한 개수가 제공한 샘플보다 10개 이상 적을때
            if (df_parsed['기상시간'].isna().sum() > 10) | ((len(sub1)-len(df_parsed)) > 10):
                print(f"[RETRY] 결측치 많음 → subject_id: {subject_id} (attempt {attempt+1})")
                continue

            parsed_results.append(df_parsed)
            break  # 성공하면 반복 종료

        except Exception as e:
            print(f"[ERROR] Parsing failed for subject {subject_id} (attempt {attempt+1}): {e}")
            if attempt == 1:
                print(f"[FAIL] 최종 실패 → subject_id: {subject_id}")
            import time
            time.sleep(2)  # 재시도 전에 대기

# 모든 subject 결과 병합
data_filled_na = pd.concat(parsed_results, ignore_index=True)
data_filled_na = data_filled_na.rename(columns={'취침시간':'취침시간_llm','기상시간':'기상시간_llm'})
data_filled_na.to_excel(f'{save_path}/{dataname}_llm_{version}.xlsx',index=False)

# check
data_filled_na.head()